# Объединение коэффициентов вариации деформации ЛЖ

Этот ноутбук выполняет слияние двух разрозненных признаков дисперсии деформации в единый признак `merged_lv_deformation_cv_pct`:
1. `dicor_cv_sfu_at_ksk_lv_pct` (CV СФУ)
2. `echo_lv_strain_plus_cv_pct` (CV GLS)

Обе переменные отражают физиологическую вариативность деформации сегментов левого желудочка. Их объединение обосновано статистической неразличимостью их распределений (U-критерий Манна-Уитни p=0.979). Перед слиянием к значениям применяется модуль `abs()`, чтобы нивелировать любые артефакты от знака минус, возникающие при расчете стрейна.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import mannwhitneyu, ks_2samp

DATASET_PATH = Path('../data/final_analysis_dataset_ver2.xlsx')
BACKUP_PATH = Path('../data/interim/final_analysis_dataset_ver2_backup_cv.xlsx')

# Загрузка датасета и создание бэкапа
df = pd.read_excel(DATASET_PATH)
df.to_excel(BACKUP_PATH, index=False)
print(f"Резервная копия сохранена в {BACKUP_PATH}")

In [ ]:
# Целевые колонки
col_sfu = 'dicor_cv_sfu_at_ksk_lv_pct'
col_gls = 'echo_lv_strain_plus_cv_pct'
col_merged = 'merged_lv_deformation_cv_pct'

# Статистическое сравнение распределений
sfu_vals = df[col_sfu].dropna()
gls_vals = df[col_gls].dropna()

print("=== Статистика до объединения ===")
print(f"{col_sfu}: {len(sfu_vals)} значений, среднее = {sfu_vals.mean():.2f}")
print(f"{col_gls}: {len(gls_vals)} значений, среднее = {gls_vals.mean():.2f}")

mw_res = mannwhitneyu(sfu_vals, gls_vals)
ks_res = ks_2samp(sfu_vals, gls_vals)
print("\n=== Проверка сходства распределений ===")
print(f"Mann-Whitney U: p-value = {mw_res.pvalue:.3f}")
print(f"Kolmogorov-Smirnov: p-value = {ks_res.pvalue:.3f}")
print("Высокие p-value (>0.05) подтверждают, что распределения статистически идентичны и могут быть объединены.")

In [ ]:
# Объединение с взятием модуля
# combine_first использует значения из col_sfu, а там где пропуски - из col_gls
df[col_merged] = df[col_sfu].combine_first(df[col_gls]).abs()

print("\n=== Статистика объединенной переменной ===")
print(f"Количество непустых значений: {df[col_merged].notna().sum()}")
print(df[col_merged].describe())

In [ ]:
# Сохранение обновленного датасета
df.to_excel(DATASET_PATH, index=False)
print(f"\nДатасет успешно обновлен и сохранен по пути {DATASET_PATH}")